# 模型调用 Invocation
创建model

In [2]:
import os

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

model = init_chat_model(
    model="qwen3.5-plus",
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    temperature=0.7,
    timeout=30,
    max_tokens=1000,
)


## invoke
1. 传单个消息

In [4]:
response = model.invoke("你是谁？")
print(response)

content='你好！我是 **Qwen3.5**，阿里巴巴最新推出的通义千问大语言模型。我基于更全面的知识语料训练，具备以下核心能力：\n\n- **超长上下文**：支持 256K 上下文窗口，能精准处理长文档、视频或复杂任务；\n- **多语言覆盖**：流畅支持全球 100+ 语言，包括中文、英语、小语种等；\n- **深度推理**：在数学、逻辑、科学领域可拆解复杂问题，提供分步解析；\n- **代码全栈**：从生成、调试到部署，支持多种编程语言，甚至能直接运行前端页面；\n- **智能体协作**：可自主规划任务、调用工具，完成多步骤操作（如数据分析、跨设备控制）；\n- **专业领域优化**：在医疗、法律等垂直场景提供符合行业规范的建议；\n- **知识时效性**：训练数据截止至 **2026 年**，能回答最新趋势和技术问题。\n\n无论是写故事、解方程、分析图表，还是帮你规划旅行路线，我都能高效协助！有什么具体任务需要我帮忙吗？ 😊' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 570, 'prompt_tokens': 12, 'total_tokens': 582, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 313, 'rejected_prediction_tokens': None, 'text_tokens': 570}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 12}}, 'model_provider': 'openai', 'model_name': 'qwen3.5-plus', 'system_fingerprint': None, 'id': 'chatcmpl-b343f316-091d-9a9d-8b48-8c8847a135d8', 'finish_reason': 'stop', 'lo

2. 传消息列表

In [ ]:

conversation = [
    {"role": "system", "content": "你是一个将汉语翻译英语的有用助手。"},
    {"role": "user", "content": "翻译：我喜欢编程。"},
    {"role": "assistant", "content": "I like programming"},
    {"role": "user", "content": "翻译：我喜欢构建应用程序。"}
]

response = model.invoke(conversation)
print(response.content)  # I like building applications.

I like building applications.


* 引入封装好的消息类

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

conversation = [
    SystemMessage("你是一个将汉语翻译英语的有用助手。"),
    HumanMessage("翻译：我喜欢编程。"),
    AIMessage("I like programming"),
    HumanMessage("翻译：我喜欢构建应用程序。")
]

response = model.invoke(conversation)
print(response.content)  # I like building applications.

content='I like building applications.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 1430, 'prompt_tokens': 48, 'total_tokens': 1478, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 1420, 'rejected_prediction_tokens': None, 'text_tokens': 1430}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 48}}, 'model_provider': 'openai', 'model_name': 'qwen3.5-plus', 'system_fingerprint': None, 'id': 'chatcmpl-349e1454-d10a-99e6-a46a-d1eeae4d3579', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d56be-3e00-7722-bb00-4ffda399a32d-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 48, 'output_tokens': 1430, 'total_tokens': 1478, 'input_token_details': {}, 'output_token_details': {'reasoning': 1420}}
I like building applications.


## stream

In [ ]:
for chunk in model.stream("My name is Webster,do you know what it means？"):
    if chunk.content:
        print(chunk.content, end="",flush=True)
        

Hello Webster! Nice to meet you.

Yes, I do know the meaning of your name. **Webster** is an English occupational surname that originally meant **"weaver."**

Here are a few details about it:

*   **Origin:** It comes from the Old English word *webbe* (meaning "weaver") combined with the suffix *-ster*, which indicates a person who performs a specific trade or activity.
*   **History:** It was originally given to someone whose job was weaving cloth.
*   **Famous Association:** The most famous bearer of the name is **Noah Webster**, the American lexicographer who published the first American dictionary (which is why we often call dictionaries "Webster's").

It's a strong name with a lot of history behind it!

In [13]:
full = None  # None | AIMessageChunk
for chunk in model.stream("What color is the sky?"):
    full = chunk if full is None else full + chunk
    print(full.content, end="", flush=True)

TheThe sky is typically **The sky is typically **blue** during theThe sky is typically **blue** during the day. ThisThe sky is typically **blue** during the day. This happens because of aThe sky is typically **blue** during the day. This happens because of a phenomenon called RayThe sky is typically **blue** during the day. This happens because of a phenomenon called Rayleigh scattering, whereThe sky is typically **blue** during the day. This happens because of a phenomenon called Rayleigh scattering, where theThe sky is typically **blue** during the day. This happens because of a phenomenon called Rayleigh scattering, where the Earth's atmosphere scThe sky is typically **blue** during the day. This happens because of a phenomenon called Rayleigh scattering, where the Earth's atmosphere scattersThe sky is typically **blue** during the day. This happens because of a phenomenon called Rayleigh scattering, where the Earth's atmosphere scatters shorter wavelengths of sunlightThe sky is typ

## batch
默认情况下，batch() 只会返回整个批次的最终输出

In [15]:
responses = model.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
])
for response in responses:
    print(response)

content='There is no single reason why parrots have colorful feathers. Instead, it is a combination of evolutionary advantages, biological chemistry, and environmental factors.\n\nHere are the primary reasons behind the vibrant plumage of parrots:\n\n### 1. Camouflage (Surprisingly)\nWhile a bright red or blue parrot stands out against a white wall, they are often well-camouflaged in their natural habitat.\n*   **The Green Canopy:** Many parrots are predominantly green, which blends perfectly with the leaves of the rainforest canopy where they live.\n*   **Dappled Light:** In a dense forest, sunlight filters through the leaves in patches of bright color and shadow. Bright reds, yellows, and blues can actually break up the bird\'s outline, making it harder for predators to spot them among colorful flowers and fruits.\n*   **Distance:** From a distance, bright colors tend to blur together. A flock of green parrots with red wing patches may look like a mass of leaves and shadows to a pred

如果你想在每个单独输入生成完成后接收输出，你可以使用 batch_as_completed() 流式传输结果：
* 结果会乱序到达，但是都是存在索引的，可以在需要时匹配，重新排序。

In [14]:
for response in model.batch_as_completed([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
]):
    print(response)

(2, AIMessage(content='At its simplest, **quantum computing is a type of computation that harnesses the unique laws of quantum mechanics** (the physics of the very small) to process information.\n\nWhile classical computers (like your laptop or phone) process information in binary bits, quantum computers use **quantum bits, or qubits**. This fundamental difference allows them to solve certain complex problems exponentially faster than the most powerful supercomputers we have today.\n\nHere is a breakdown of how it works, why it matters, and where we stand.\n\n---\n\n### 1. The Core Difference: Bits vs. Qubits\nTo understand quantum computing, you first need to understand classical computing.\n*   **Classical Bit:** Think of a light switch. It is either **Off (0)** or **On (1)**. Every app, photo, and website on your computer is ultimately a long string of 0s and 1s.\n*   **Quantum Qubit:** Think of a spinning coin. While it is spinning, it is technically both **Heads and Tails** at the